## creating model (stress level):

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import os

# Load the dataset
df = pd.read_csv("wheat_dataset.csv")

# Features and target
X = df.drop(columns=["timestamp", "irrigation_amount", "stress_level"])
y = df["stress_level"]

# Encode categorical target
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # healthy=0, moderate=1, stressed=2

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler and label encoder for future use
joblib.dump(scaler, 'stress_scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("Scaler saved as 'stress_scaler.pkl' and label encoder saved as 'label_encoder.pkl'")

# Initialize and train logistic regression
clf = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
clf.fit(X_train_scaled, y_train)

# Save the model
joblib.dump(clf, 'stress_model.pkl')
print("Model saved as 'stress_model.pkl'")

# Predictions and evaluation
y_pred = clf.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))


Scaler saved as 'stress_scaler.pkl' and label encoder saved as 'label_encoder.pkl'
Model saved as 'stress_model.pkl'
Accuracy: 0.9822530864197531

Confusion Matrix:
 [[593   7   2]
 [  2 170   3]
 [  6   3 510]]

Classification Report:
               precision    recall  f1-score   support

     healthy       0.99      0.99      0.99       602
    moderate       0.94      0.97      0.96       175
    stressed       0.99      0.98      0.99       519

    accuracy                           0.98      1296
   macro avg       0.97      0.98      0.98      1296
weighted avg       0.98      0.98      0.98      1296



c:\Users\PC\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


## creating model (irrigation):

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

# Load dataset
df = pd.read_csv("wheat_dataset.csv")

# Features and target
X = df[["temperature", "humidity", "soil_moisture", "soil_temperature", "light_intensity"]]
y = df["irrigation_amount"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for future use
joblib.dump(scaler, 'irrigation_scaler.pkl')
print("Scaler saved as 'irrigation_scaler.pkl'")

# Stronger Random Forest
rf_reg = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_leaf=2,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

# Train
rf_reg.fit(X_train_scaled, y_train)

# Save the model
joblib.dump(rf_reg, 'irrigation_model.pkl')
print("Model saved as 'irrigation_model.pkl'")

# Predictions and evaluation
y_pred = rf_reg.predict(X_test_scaled)
print(f"Mean Squared Error (MSE): {mean_squared_error(y_test, y_pred):.2f}")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R^2 Score: {r2_score(y_test, y_pred):.2f}")


Scaler saved as 'irrigation_scaler.pkl'
Model saved as 'irrigation_model.pkl'
Mean Squared Error (MSE): 30.19
Mean Absolute Error (MAE): 4.66
R^2 Score: 0.80
